# 18. Conferindo o resultado por programação dinâmica (QuantEcon)

Neste notebook eu resolvo o mesmo modelo de duas maneiras diferentes e comparo o
resultado das duas (requisitos F6, F8 e F11).

1. Do jeito do projeto, chamando direto o app.nucleo (resolver_alpha_otimo,
   phi_chapeu, recorrencia_A, fracoes_consumo e funcao_valor). Não reescrevi
   nada aqui: o que está sendo testado é o módulo que a aplicação usa mesmo.
2. Na força bruta, resolvendo a equação de Bellman por iteração da função valor
   numa grade de riqueza, e usando a quadratura do QuantEcon para as
   esperanças. Escrevi essa parte sem olhar o app/, senão a comparação não
   valeria de nada, porque estaria comparando o código com ele mesmo.

Se os dois caminhos derem o mesmo número, o código do projeto está certo. Os
parâmetros aqui são só ilustrativos: gamma = 3, beta = 0.96 e T = 5.

In [1]:
import sys, os
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np, time
from quantecon.quad import qnwnorm  # quadratura gaussiana
from scipy.optimize import minimize_scalar
from scipy.interpolate import PchipInterpolator
from app import nucleo

In [2]:
gamma = 3.0
beta = 0.96
T = 5
mu_g = 1.02
Rf = 1.008
sigma = 0.08

# Discretiza a distribuicao dos retornos por quadratura (QuantEcon)
Rn, w = qnwnorm(15, mu_g, sigma**2)
Rn = np.maximum(Rn, 0.0); w = w / w.sum()  # responsabilidade limitada + normaliza

print('nos de retorno:', Rn.round(3))
print('pesos somam:', round(float(w.sum()), 6))

nos de retorno: [0.511 0.605 0.684 0.757 0.825 0.891 0.956 1.02  1.084 1.149 1.215 1.283
 1.356 1.435 1.529]
pesos somam: 1.0


## 1. A solução do projeto (app/nucleo.py)

Aqui é só chamar as funções do módulo da aplicação.

Tem um detalhe para os dois lados resolverem a mesma distribuição. O app.nucleo
trabalha com cenários que valem todos a mesma coisa, porque a esperança dele é
uma média simples. Já a quadratura dá um peso diferente para cada nó. Então eu
sorteio os nós de novo respeitando esses pesos, com semente fixa para o
resultado não mudar de uma execução para outra.

Sobrando só o erro de sorteio, ele fica meio amplificado aqui. Como o alpha é
mais ou menos (mu - R_f) / (gamma * sigma ao quadrado), com esses valores um
erro pequeno na média vira um erro bem maior no alpha. Por isso a
tolerância lá na comparação está mais folgada do que se esperaria.

In [3]:
N_MC = 2_000_000
R_mc = np.random.default_rng(1).choice(Rn, size=N_MC, p=w).reshape(-1, 1)  # semente fixa (NF4)

alpha_an = nucleo.resolver_alpha_otimo(R_mc, Rf, gamma)[0]  # FOC (Monte Carlo)
Phi = nucleo.phi_chapeu(np.array([alpha_an]), R_mc, Rf, gamma)
A = nucleo.recorrencia_A(Phi, beta, gamma, T)  # daqui para baixo e conta exata, sem sorteio
theta_an = nucleo.fracoes_consumo(A, gamma)
V_an = nucleo.funcao_valor(A, 1.0, gamma)  # V_t(W=1)

print('alpha* =', round(alpha_an, 4), '| Phi =', round(Phi, 4))
print('theta  =', theta_an[:T].round(4))
print('residuo da FOC na amostra:', f'{nucleo.funcao_foc(np.array([alpha_an]), R_mc, Rf, gamma)[0]:.2e}')

alpha* = 0.628 | Phi = 0.9769
theta  = [0.1757 0.2087 0.2581 0.3405 0.5054]
residuo da FOC na amostra: -4.04e-15


## 2. A solução por programação dinâmica (iteração da função valor)

Resolve a equação de Bellman de trás para frente, numa grade de valores de
riqueza, sem usar a forma fechada em momento nenhum. A esperança é feita pela
quadratura do QuantEcon. A otimização em consumo e em alpha é por descida de
coordenadas, começando de um alpha de 0.5, que é um chute neutro e não a
resposta esperada. A função valor entre os pontos da grade é interpolada com
PCHIP, que é uma cúbica que não cria ondulação onde não tem.

In [4]:
EPS = 1e-10
def u(c):
    return c**(1 - gamma)/(1 - gamma)
grade = np.geomspace(0.05, 40.0, 220)
wmin = grade[0]
wmax = grade[-1]
V = u(grade)  # condicao terminal V_T(W) = u(W)
theta_vfi = np.zeros(T)
alpha_vfi = np.zeros(T)
V_vfi = np.zeros(T)

In [5]:
def resolver_ponto(W, interp):
    def menos_valor(x):
        s, al = x; c = W - s
        if c <= EPS or s <= EPS: return 1e18
        Rp = np.maximum(Rf + al*(Rn - Rf), EPS)
        Wp = np.clip(s*Rp, wmin, wmax)
        return -(u(c) + beta*np.sum(w*interp(Wp)))  # E[.] por quadratura
    s = 0.7*W
    al = 0.5  # alpha0 neutro
    for _ in range(8):  # descida de coordenadas
        s = minimize_scalar(lambda ss: menos_valor([ss, al]), bounds=(EPS, W - EPS), method='bounded', options={'xatol':1e-9}).x
        al = minimize_scalar(lambda aa: menos_valor([s, aa]), bounds=(-1, 4), method='bounded', options={'xatol':1e-7}).x
    return s, al, -menos_valor([s, al])

In [6]:
t0 = time.time()
for t in range(T - 1, -1, -1):  # inducao retroativa
    interp = PchipInterpolator(grade, V, extrapolate=True)
    s1, al1, v1 = resolver_ponto(1.0, interp)  # politica em W=1
    theta_vfi[t] = 1 - s1; alpha_vfi[t] = al1; V_vfi[t] = v1
    V = np.array([resolver_ponto(W, interp)[2] for W in grade])  # V_t sobre a grade

print('VFI resolvido em', round(time.time() - t0, 1), 's')
print('alpha* VFI =', alpha_vfi.round(4))
print('theta  VFI =', theta_vfi.round(4))

VFI resolvido em 25.4 s
alpha* VFI = [0.6289 0.6283 0.6282 0.6283 0.6282]
theta  VFI = [0.1757 0.2087 0.2581 0.3405 0.5053]


## 3. Comparação

In [7]:
print('%-8s %14s %18s' % ('', 'app/nucleo', 'VFI (QuantEcon)'))
print('%-8s %14.4f %18.4f' % ('alpha*', alpha_an, alpha_vfi.mean()))

             app/nucleo    VFI (QuantEcon)
alpha*           0.6280             0.6284


In [8]:
print(' t    theta_app   theta_VFI      V(1)_app     V(1)_VFI')
for t in range(T):
    print(' %d    %9.4f   %8.4f    %10.3f   %10.3f' % (t, theta_an[t], theta_vfi[t], V_an[t], V_vfi[t]))

 t    theta_app   theta_VFI      V(1)_app     V(1)_VFI
 0       0.1757     0.1757       -92.161      -92.159
 1       0.2087     0.2087       -55.040      -55.039
 2       0.2581     0.2581       -29.085      -29.085
 3       0.3405     0.3405       -12.666      -12.666
 4       0.5054     0.5053        -3.874       -3.874


In [9]:
# Testes: o codigo do projeto reproduz a PD independente?
assert np.allclose(alpha_vfi, alpha_an, atol=5e-3), 'alpha nao bate'
assert np.allclose(theta_vfi, theta_an[:T], rtol=1e-2), 'theta nao bate'
assert np.allclose(V_vfi, V_an[:T], rtol=1e-2), 'V nao bate'

In [10]:
# Observacao (nao e requisito): o alpha da PD sai constante no tempo, que e a miopia
assert np.allclose(alpha_vfi, alpha_vfi[0], atol=5e-3), 'alpha deveria ser constante (miopia)'

## Conclusão

Os dois caminhos deram o mesmo resultado no alpha, nas frações de consumo e na
função valor. Como a programação dinâmica foi escrita sem usar nada do app/,
o que fica conferido é o código que a aplicação executa. Só como observação,
que não é requisito do projeto: o alpha sai constante no tempo mesmo sem
ninguém ter obrigado ele a isso, que é a miopia.

As frações de consumo e a função valor batem até a terceira ou quarta casa
decimal. O que sobra de diferença vem do erro de amostragem do alpha e do phi,
do lado do projeto, e da discretização da grade e da interpolação, do lado da
programação dinâmica.

A programação dinâmica resolve o modelo em geral. A forma fechada só existe
porque, com utilidade CRRA e retornos i.i.d., o alpha e o Phi são os mesmos em
todos os períodos.